In [ ]:
import pandas as pd
import numpy as np

In [ ]:
employee = pd.read_csv("../data/raw/employee_data 1.csv")
attrition = pd.read_csv("../data/raw/Attrition 1.csv")
performance = pd.read_csv("../data/raw/employee_performance_data 1.csv")

print("Employee Data:", employee.shape)
print("Attrition Data:", attrition.shape)
print("Performance Data:", performance.shape)

In [ ]:
print("EMPLOYEE DATA")
display(employee.head())

print("ATTRITION DATA")
display(attrition.head())

print("EMPLOYEE PERFORMANCE DATA")
display(performance.head())

In [ ]:
print("Employee Data Columns:")
print(employee.columns.tolist())

print("\nAttrition Data Columns:")
print(attrition.columns.tolist())

print("\nPerformance Data Columns:")
print(performance.columns.tolist())

In [ ]:
print("EMPLOYEE DATA")
employee.info()

print("\nATTRITION DATA")
attrition.info()

print("\nPERFORMANCE DATA")
performance.info()

In [ ]:
print("EMPLOYEE DATA")
display(pd.DataFrame({
    "Column": employee.columns,
    "Data Type": employee.dtypes.astype(str),
    "Non-Null": employee.notnull().sum().values
}))

print("ATTRITION DATA")
display(pd.DataFrame({
    "Column": attrition.columns,
    "Data Type": attrition.dtypes.astype(str),
    "Non-Null": attrition.notnull().sum().values
}))

print("PERFORMANCE DATA")
display(pd.DataFrame({
    "Column": performance.columns,
    "Data Type": performance.dtypes.astype(str),
    "Non-Null": performance.notnull().sum().values
}))

Upto this data profiling is done

Section 6 → Data Quality Assessment → Missing Value Analysis down from here

In [ ]:
print("Missing Values - Employee Data")
print(employee.isnull().sum())

print("\nMissing Values - Attrition Data")
print(attrition.isnull().sum())

print("\nMissing Values - Performance Data")
print(performance.isnull().sum())

In [ ]:
print("Duplicate rows in Employee Data:", employee.duplicated().sum())
print("Duplicate rows in Attrition Data:", attrition.duplicated().sum())
print("Duplicate rows in Performance Data:", performance.duplicated().sum())

In [ ]:
duplicates = attrition[attrition.duplicated(keep=False)]

print("Duplicate Attrition Records:")
display(duplicates)

In [ ]:
attrition = attrition.drop_duplicates().reset_index(drop=True)

print("Attrition Data Shape After Removing Duplicates:", attrition.shape)
print("Duplicate rows remaining:", attrition.duplicated().sum())

Section 5 → Data Profiling → Statistical Summary

In [ ]:
print("EMPLOYEE DATA - NUMERICAL SUMMARY")
display(employee.describe())

print("PERFORMANCE DATA - NUMERICAL SUMMARY")
display(performance.describe())

In [ ]:
print("Gender values:")
print(employee["gender"].value_counts())

print("\nGender column values:")
print(employee["Gender"].value_counts())

print("\nDepartments:")
print(employee["Department"].value_counts())

print("\nJob Roles:")
print(employee["Job_Role"].value_counts())

print("\nEducation Levels:")
print(employee["Education_Level"].value_counts())

print("\nMarital Status:")
print(employee["Marital_Status"].value_counts())

In [ ]:
print("Gender vs Gender Column:")
display(pd.crosstab(employee["gender"], employee["Gender"]))

In [ ]:
print("Attrition Distribution:")
print(attrition["attrition"].value_counts())

print("\nAttrition Percentage:")
print(attrition["attrition"].value_counts(normalize=True) * 100)

print("\nExit Interview Score:")
print(attrition["Exit_Interview_Score"].value_counts().sort_index())

In [ ]:
print("Performance Rating:")
print(performance["Performance_Rating"].value_counts().sort_index())

print("\nWork-Life Balance:")
print(performance["Work_Life_Balance"].value_counts().sort_index())

print("\nJob Satisfaction:")
print(performance["Job_Satisfaction"].value_counts().sort_index())

print("\nLast Promotion Year:")
print(performance["Last_Promotion_Year"].value_counts().sort_index())

In [ ]:
employee_ids = set(employee["Employee_ID"])
performance_ids = set(performance["Employee_ID"])
attrition_ids = set(attrition["employee_ID"])

print("Attrition IDs not found in Employee Data:",
      len(attrition_ids - employee_ids))

print("Attrition IDs not found in Performance Data:",
      len(attrition_ids - performance_ids))

print("Employee IDs not found in Performance Data:",
      len(employee_ids - performance_ids))

In [ ]:
print("Unique Employee IDs:", employee["Employee_ID"].nunique())
print("Unique Performance IDs:", performance["Employee_ID"].nunique())
print("Unique Attrition IDs:", attrition["employee_ID"].nunique())

In [ ]:
attrition_id_counts = attrition["employee_ID"].value_counts()

print("Repeated Employee IDs in Attrition Data:")
display(attrition_id_counts[attrition_id_counts > 1])

In [ ]:
repeated_ids = attrition_id_counts[attrition_id_counts > 1].index

repeated_attrition = attrition[
    attrition["employee_ID"].isin(repeated_ids)
].sort_values("employee_ID")

print("Repeated Attrition Records:")
display(repeated_attrition)

In [ ]:
repeated_analysis = (
    repeated_attrition
    .groupby("employee_ID")
    .agg(
        Record_Count=("employee_ID", "size"),
        Unique_Attrition=("attrition", "nunique"),
        Unique_Interview_Scores=("Exit_Interview_Score", "nunique")
    )
    .reset_index()
)

print("Analysis of Repeated Attrition IDs:")
display(repeated_analysis)

In [ ]:
conflicting_attrition = repeated_analysis[
    repeated_analysis["Unique_Attrition"] > 1
]

same_attrition = repeated_analysis[
    repeated_analysis["Unique_Attrition"] == 1
]

print("Repeated IDs with conflicting Attrition values:",
      len(conflicting_attrition))

print("Repeated IDs with same Attrition value but different scores:",
      len(same_attrition))

In [ ]:
conflicting_ids = conflicting_attrition["employee_ID"].tolist()

conflicting_records = attrition[
    attrition["employee_ID"].isin(conflicting_ids)
].sort_values("employee_ID")

print("Conflicting Attrition Records:")
display(conflicting_records)

In [ ]:
attrition_clean = attrition.copy()

# Remove records with conflicting attrition
attrition_clean = attrition_clean[
    ~attrition_clean["employee_ID"].isin(conflicting_ids)
].copy()
same_status_repeated_ids = same_attrition["employee_ID"].tolist()

for emp_id in same_status_repeated_ids:
    attrition_clean.loc[
        attrition_clean["employee_ID"] == emp_id,
        "Exit_Interview_Score"
    ] = np.nan

print("Original Attrition rows:", len(attrition))
print("Cleaned Attrition rows:", len(attrition_clean))
print("Unique Employee IDs:", attrition_clean["employee_ID"].nunique())
print("Missing Exit Interview Scores:",
      attrition_clean["Exit_Interview_Score"].isna().sum())
print("Duplicate Employee IDs remaining:",
      attrition_clean["employee_ID"].duplicated().sum())

In [ ]:
attrition_clean = (
    attrition_clean
    .drop_duplicates(subset=["employee_ID"], keep="first")
    .reset_index(drop=True)
)

print("Final Cleaned Attrition Rows:", len(attrition_clean))
print("Unique Employee IDs:", attrition_clean["employee_ID"].nunique())
print("Duplicate Employee IDs remaining:",
      attrition_clean["employee_ID"].duplicated().sum())
print("Missing Exit Interview Scores:",
      attrition_clean["Exit_Interview_Score"].isna().sum())

In [ ]:
print("Employee ID duplicates:",
      employee["Employee_ID"].duplicated().sum())

print("Performance ID duplicates:",
      performance["Employee_ID"].duplicated().sum())

In [ ]:
print("Department values:")
print(employee["Department"].value_counts())

print("\nJob Role values:")
print(employee["Job_Role"].value_counts())

print("\nEducation Level values:")
print(employee["Education_Level"].value_counts())

print("\nMarital Status values:")
print(employee["Marital_Status"].value_counts())

In [ ]:
print("Age outside 18-65:", ((employee["Age"] < 18) | (employee["Age"] > 65)).sum())

print("Job Tenure outside 0-40:",
      ((employee["Job_Tenure"] < 0) | (employee["Job_Tenure"] > 40)).sum())

print("Distance From Home <= 0:",
      (employee["Distance_From_Home"] <= 0).sum())

print("Performance Rating outside 1-5:",
      ((performance["Performance_Rating"] < 1) |
       (performance["Performance_Rating"] > 5)).sum())

print("Work-Life Balance outside 1-5:",
      ((performance["Work_Life_Balance"] < 1) |
       (performance["Work_Life_Balance"] > 5)).sum())

print("Job Satisfaction outside 1-5:",
      ((performance["Job_Satisfaction"] < 1) |
       (performance["Job_Satisfaction"] > 5)).sum())

print("Training Hours outside 0-300:",
      ((performance["Training_Hours"] < 0) |
       (performance["Training_Hours"] > 300)).sum())

In [ ]:
print("Promotion Year Range:")
print("Minimum Year:", performance["Last_Promotion_Year"].min())
print("Maximum Year:", performance["Last_Promotion_Year"].max())

print("\nInvalid Promotion Years:")
print(
    ((performance["Last_Promotion_Year"] < 2000) |
     (performance["Last_Promotion_Year"] > 2026)).sum()
)

In [ ]:
employee_clean = employee.copy()
performance_clean = performance.copy()

print("Employee Clean Shape:", employee_clean.shape)
print("Performance Clean Shape:", performance_clean.shape)

In [ ]:
employee_clean["Tenure_Category"] = pd.cut(
    employee_clean["Job_Tenure"],
    bins=[0, 5, 10, 15, 20],
    labels=["0-5 Years", "6-10 Years", "11-15 Years", "16-20 Years"],
    include_lowest=True
)

print("Tenure Category Distribution:")
print(employee_clean["Tenure_Category"].value_counts().sort_index())

In [ ]:
reference_year = performance_clean["Last_Promotion_Year"].max()

performance_clean["Years_Since_Promotion"] = (
    reference_year - performance_clean["Last_Promotion_Year"]
)

print("Reference Year:", reference_year)
print("\nYears Since Promotion:")
print(performance_clean["Years_Since_Promotion"].describe())

In [ ]:
performance_clean["Overall_Satisfaction"] = (
    performance_clean["Work_Life_Balance"] +
    performance_clean["Job_Satisfaction"]
) / 2

print("Overall Satisfaction Summary:")
print(performance_clean["Overall_Satisfaction"].describe())

In [ ]:
performance_clean["Performance_Category"] = pd.cut(
    performance_clean["Performance_Rating"],
    bins=[0, 2, 3, 5],
    labels=["Low", "Average", "High"],
    include_lowest=True
)

print("Performance Category Distribution:")
print(performance_clean["Performance_Category"].value_counts().sort_index())

In [ ]:
employee_performance = pd.merge(
    employee_clean,
    performance_clean,
    on="Employee_ID",
    how="left"
)

final_data = pd.merge(
    employee_performance,
    attrition_clean,
    left_on="Employee_ID",
    right_on="employee_ID",
    how="left"
)

print("Final Dataset Shape:", final_data.shape)
print("Employees with Attrition Record:",
      final_data["employee_ID"].notna().sum())
print("Employees without Attrition Record:",
      final_data["employee_ID"].isna().sum())

In [ ]:
print("Final Dataset Shape:", final_data.shape)

print("\nDuplicate Employee IDs:")
print(final_data["Employee_ID"].duplicated().sum())

print("\nMissing Values in Final Dataset:")
print(final_data.isnull().sum())

Loading part

In [ ]:
employee_clean.to_csv(
    "../data/processed/employee_clean.csv",
    index=False
)

performance_clean.to_csv(
    "../data/processed/performance_clean.csv",
    index=False
)

attrition_clean.to_csv(
    "../data/processed/attrition_clean.csv",
    index=False
)

final_data.to_csv(
    "../data/processed/employee_performance_attrition.csv",
    index=False
)

print("Processed datasets saved successfully.")

In [ ]:
missing_score_records = attrition_clean[
    attrition_clean["Exit_Interview_Score"].isna()
]

print("Records with missing Exit Interview Score:")
display(missing_score_records)